# Apprentissage Non Supervisé - Villes Françaises

**Dataset : villes.csv**

32 villes françaises décrites par les températures moyennes des 12 mois de l'année.

---

## Configuration et Imports

In [ ]:
from utils_ans import *
setup_environment()

---
# I. Réduction de dimensions et Visualisation des données
---

## 1. Importation du jeu de données

In [ ]:
data = load_data('./data/villes.csv')

In [ ]:
X, labels, feature_names = prepare_data(data, feature_cols=slice(1, 13), label_col=0)

## 2. Analyse en Composantes Principales (ACP)

In [ ]:
# Standardisation et ACP
X_scaled, X_pca, pca, scaler = perform_pca(X)

In [ ]:
cumulative = print_variance_explained(pca)

In [ ]:
plot_variance(pca)

### Réponse : Nombre d'axes à retenir

**Analyse :**
- La première composante principale explique environ **97%** de la variance totale
- Les deux premières composantes expliquent plus de **99%** de la variance

**Conclusion : 2 axes suffisent** pour conserver une excellente représentation de l'information.

Cela s'explique par le fait que les températures mensuelles sont très corrélées entre elles.

In [ ]:
loadings_df = get_loadings(pca, feature_names, n_components=2)

### Interprétation des deux premiers axes principaux

**Axe 1 (PC1) - "Axe de la température générale"** :
- Toutes les variables ont des corrélations positives et similaires
- Représente le **niveau moyen de température** sur l'année
- Villes à droite = globalement plus chaudes (Sud)
- Villes à gauche = globalement plus froides (Nord/Est)

**Axe 2 (PC2) - "Axe de la continentalité/amplitude thermique"** :
- Les mois d'hiver ont des corrélations opposées aux mois d'été
- Représente le **contraste été/hiver**
- Villes en haut = climat océanique (faible amplitude)
- Villes en bas = climat continental (forte amplitude)

In [ ]:
plot_correlation_circle(pca, feature_names)

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=60, alpha=0.7)

for label, x, y in zip(labels, X_pca[:, 0], X_pca[:, 1]):
    plt.annotate(str(label), xy=(x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Projection des villes françaises dans le plan principal')
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.show()

### Analyse de la visualisation

On observe une **segmentation géographique** claire :
- **À droite** (PC1 > 0) : Villes du sud (Perpignan, Marseille, Nice, Toulouse, Ajaccio...)
- **À gauche** (PC1 < 0) : Villes du nord et de l'est (Strasbourg, Nancy, Lille, Embrun...)
- **En haut** (PC2 > 0) : Villes à climat océanique (Brest, Nantes, Rennes...)
- **En bas** (PC2 < 0) : Villes à climat continental ou montagnard (Strasbourg, Embrun, Grenoble...)

---
# II. Clustering
---

## 1. KMeans (3 clusters)

In [ ]:
# Application de KMeans
clustering_kmeans, kmeans = apply_kmeans(X_scaled, n_clusters=3)

print("Clusters (3) :")
print("=" * 60)
for i in range(3):
    members = labels[clustering_kmeans == i]
    print(f"\nCluster {i} ({len(members)} éléments) :")
    print(f"  {list(members)}")

In [ ]:
plot_clustering(X_pca, clustering_kmeans, labels, 'Clustering KMeans (K=3) - Villes françaises')

## 2. AgglomerativeClustering avec différentes méthodes

In [ ]:
# Single linkage
clustering_single = apply_agglomerative(X_scaled, n_clusters=3, linkage='single')

print("Clusters Single (3) :")
for i in range(3):
    members = labels[clustering_single == i]
    print(f"Cluster {i} ({len(members)} éléments): {list(members)}")

plot_clustering(X_pca, clustering_single, labels, 'AgglomerativeClustering - Single Linkage')

In [ ]:
# Ward linkage
clustering_ward = apply_agglomerative(X_scaled, n_clusters=3, linkage='ward')

print("Clusters Ward (3) :")
for i in range(3):
    members = labels[clustering_ward == i]
    print(f"Cluster {i} ({len(members)} éléments): {list(members)}")

plot_clustering(X_pca, clustering_ward, labels, 'AgglomerativeClustering - Ward Linkage')

In [ ]:
# Average linkage
clustering_average = apply_agglomerative(X_scaled, n_clusters=3, linkage='average')

print("Clusters Average (3) :")
for i in range(3):
    members = labels[clustering_average == i]
    print(f"Cluster {i} ({len(members)} éléments): {list(members)}")

plot_clustering(X_pca, clustering_average, labels, 'AgglomerativeClustering - Average Linkage')

## 3. Détermination du nombre optimal de clusters (Silhouette)

In [ ]:
# Calcul des scores Silhouette
scores, best_k = compute_silhouette_scores(X_scaled)

In [ ]:
plot_silhouette_scores(scores)

### Note sur le nombre optimal de clusters

L'indice Silhouette indique K=2 comme optimal, suggérant une séparation naturelle **Nord/Sud**.

Cependant, l'exercice demande 3 clusters, ce qui permet une segmentation plus fine :
- Villes du Sud (méditerranéennes)
- Villes du Centre/Ouest (océaniques)
- Villes du Nord/Est (continentales)

## 4. Comparaison des méthodes pour 3 clusters

In [ ]:
results = compare_methods(X_scaled, n_clusters=3)

### Commentaire

**Ward et KMeans** donnent généralement les meilleurs résultats car ils minimisent l'inertie intra-classe.

**Single linkage** crée souvent des clusters très déséquilibrés (effet de chaînage).

## 5. Avantages et inconvénients des méthodes

### Classification Hiérarchique (AgglomerativeClustering)

**Avantages :**
- Pas besoin de spécifier K à l'avance
- Produit une hiérarchie complète (dendrogramme)
- Résultats déterministes
- Peut découvrir des clusters de formes arbitraires (avec single linkage)

**Inconvénients :**
- Complexité O(n²) ou O(n³)
- Ne passe pas à l'échelle pour grands datasets
- Décisions de fusion irréversibles
- Single linkage : effet de chaînage

---

### Partitionnement (KMeans)

**Avantages :**
- Très rapide O(n×K×iterations)
- Scalable pour grands datasets
- Clusters compacts et bien séparés
- Simple à comprendre et implémenter

**Inconvénients :**
- Nécessite K à l'avance
- Sensible à l'initialisation (non-déterministe)
- Sensible aux outliers
- Assume des clusters sphériques

## 6. Approche Hybride

In [ ]:
clustering_hyb, centers = clustering_hybride(X_scaled, n_clusters=3)

score_hyb = metrics.silhouette_score(X_scaled, clustering_hyb, metric='euclidean')
print(f"Approche Hybride - Silhouette : {score_hyb:.4f}")

print("\nClusters Hybride (3) :")
for i in range(3):
    members = labels[clustering_hyb == i]
    print(f"Cluster {i} ({len(members)} éléments): {list(members)}")

In [ ]:
plot_clustering(X_pca, clustering_hyb, labels, 'Approche Hybride (Ward + KMeans)')

### Explication de l'approche hybride

L'approche hybride combine :
- **Ward** : pour obtenir une initialisation déterministe et intelligente
- **KMeans** : pour permettre les réaffectations et optimiser la partition

**Avantages** :
- Résultats reproductibles
- Évite les problèmes d'initialisation aléatoire de KMeans
- Peut améliorer les résultats de Ward seul

---
# Conclusion

L'analyse des villes françaises par leurs températures révèle :

1. **ACP** : 2 composantes capturent >99% de la variance
2. **Interprétation** :
   - PC1 = température moyenne annuelle (gradient Nord-Sud)
   - PC2 = amplitude thermique (gradient Océanique-Continental)
3. **Clustering** : 3 groupes climatiques identifiables
   - Méditerranéen (Sud)
   - Océanique (Ouest)
   - Continental (Nord/Est)
4. **Méthode recommandée** : Ward ou approche hybride